In [14]:
# selenium, Beautifulsoup 임포트
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as ec
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup
import time
import csv


In [70]:
# 착한 주유소 크롤링

# 대기 시간 변수 선언
sleep_time = 2

# 데이터별 리스트 초기화(시/도 구분없이 한 리스트에 전부 저장)
name_lst = [] # 이름 리스트
address_lst = [] # 도로명 주소 리스트
g_price_lst = [] # 휘발유 가격 리스트
d_price_lst = [] # 경유 가격 리스트
phone_lst = [] # 전화번호 리스트

# selenium 드라이버 객체 생성 및 페이지 불러오기
driver = webdriver.Chrome()
driver.get('https://www.opinet.co.kr/searRgOsSelect.do')
WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))

# 시/도 select_box의 전체 옵션 수
option_cnt = len(Select(driver.find_element(By.ID, 'SIDO_NM0')).options)

# 전국 시/도 착한주유소 이름, 휘발유 가격, 경유 가격, 주소 크롤링 루프문
for i in range(1, option_cnt):
    # 시/도 선택하는 select_box 객체 생성
    select_element = driver.find_element(By.ID, 'SIDO_NM0')
    select_box = Select(select_element)

    # 시/도 선택
    select_box.select_by_index(i)

    # 조회 버튼 클릭
    driver.find_element(By.ID, 'searRgSelect').click()
    time.sleep(sleep_time) # 페이지 요소들 로딩될때까지 대기
    
    # 시의 착한 주유소 세부정보 크롤링
    # 주유소 세부정보 링크 객체 리스트
    WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.CSS_SELECTOR, '.rlist > a')))
    href_elements = driver.find_elements(By.CSS_SELECTOR, '.rlist > a')

    # 각 주유소 세부정보 링크 클릭 -> 세부정보 크롤링 루프문
    for i in range(len(href_elements)):
        try:
            # 링크 클릭
            href_elements[i].click()

            # Beautiful soup 객체 생성
            results_html = driver.page_source
            soup = BeautifulSoup(results_html, 'html.parser')

            # 주소 문자열 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'rd_addr')))
            address = soup.select_one('#rd_addr').text
            address_lst.append(address)

            # 이름 문자열 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'os_nm')))
            name = soup.select_one('#os_nm').text
            name_lst.append(name)

            # 휘발유 가격 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'b027_p')))
            g_price = soup.select_one('#b027_p').text
            g_price_lst.append(g_price)
            
            # 경유 가격 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'd047_p')))
            d_price = soup.select_one('#d047_p').text
            d_price_lst.append(d_price)

            # 전화번호 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'phn_no')))
            phone = soup.select_one('#phn_no').text
            phone_lst.append(phone)
        
        except:
            continue
        
# print(len(name_lst))
# print(len(g_price_lst))
# print(len(d_price_lst))
# print(len(address_lst))
# print(len(phone_lst))
# print(name_lst)
# print(g_price_lst)
# print(d_price_lst)
# print(address_lst)
# print(phone_lst)

# 시/도별로 저장하고 싶으면 루프마다 name_lst, g_price_lst, d_price_lst, phone_lst 초기화 후 db에 저장




In [75]:
# 착한 주유소 데이터 중복 제거

good_oil_lst = []

for nm, gp, dp, ad, pn in zip(name_lst, g_price_lst, d_price_lst, address_lst, phone_lst):
    good_oil_lst.append([nm, ad, pn, gp, dp])

name_set = set(name_lst)

name_lst_dp = list(name_set)

good_oil_lst_dp = []

for name in name_lst_dp:
    for good_oil in good_oil_lst:
        if name == good_oil[0]:
            good_oil_lst_dp.append([name] + good_oil[1:])
            break




In [78]:
# 착한 주유소 csv파일로 저장
with open('good_oil_info.csv', 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['o_name', 'o_addr', 'o_phone', 'gasoline_price', 'diesel_price'])

    for good_oil in good_oil_lst_dp:
        writer.writerow(good_oil)

In [1]:
# FAQ 크롤링

# 데이터별 리스트 초기화
question_lst = []
answer_lst = []

# selenium 드라이버 객체 생성 및 페이지 불러오기
driver = webdriver.Chrome()
driver.get('https://www.opinet.co.kr/user/cufaq/cufaqSelectPrice.do')
WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.CSS_SELECTOR, '.t_left.input > a')))

# 전체 질문 수
question_cnt = len(driver.find_elements(By.CSS_SELECTOR, '.t_left.input > a'))

for i in range(question_cnt):
    # FAQ 하나를 클릭
    question_element = driver.find_elements(By.CSS_SELECTOR, '.t_left.input > a')
    question_element[i].click()
    WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.CLASS_NAME, 'view_contents')))

    # Beautiful soup 객체 생성
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # FAQ 질문과 답변을 각각 리스트에 저장
    question_lst.append(soup.select_one('.title_txt').text)
    answer_lst.append(soup.select_one('.view_contents').text.replace('\n', '').replace('\t', ''))

    # FAQ 목록 페이지로 돌아가기
    driver.back()
    time.sleep(sleep_time)

# print(len(question_lst))
# print(len(answer_lst))
# print(question_lst)
# print(answer_lst)



NameError: name 'webdriver' is not defined

In [ ]:
# 전국 주유소 유가 크롤링

# 명시적 대기 시간
sleep_time = 0.5
# 최대 대기 시간
max_sleep_time = 60

# 데이터별 리스트 초기화(시/도 구분없이 한 리스트에 전부 저장)
all_name_lst = [] # 이름 리스트
all_address_lst = [] # 도로명 주소 리스트
all_g_price_lst = [] # 휘발유 가격 리스트
all_d_price_lst = [] # 경유 가격 리스트
all_phone_lst = [] # 전화번호 리스트

# selenium 드라이버 객체 생성 및 페이지 불러오기
driver = webdriver.Chrome()
driver.get('https://www.opinet.co.kr/searRgSelect.do')
WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))

# 시/도 selectbox의 옵션 수
sido_cnt = len(Select(driver.find_element(By.ID, 'SIDO_NM0')).options)


# 전국 주유소 이름, 휘발유 가격, 경유 가격, 주소 크롤링 루프문
for i in range(1, sido_cnt):

    # 시/도 선택하는 select_box 객체 생성
    time.sleep(sleep_time)
    sido_select_element = driver.find_element(By.ID, 'SIDO_NM0')
    sido_select_box = Select(sido_select_element)

    # 시/도 선택
    sido_select_box.select_by_index(i)
    WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIGUNGU_NM0')))
    time.sleep(sleep_time)

    # 시/군/구 selectbox의 옵션 수
    sigungu_cnt = len(Select(driver.find_element(By.ID, 'SIGUNGU_NM0')).options)

    for j in range(1, sigungu_cnt):

        # 루프를 continue로 빠져나온 후 대기 시간
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIGUNGU_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'totCnt')))

        # 시/군/구 선택하는 select_box 객체 생성
        time.sleep(sleep_time)
        sigungu_select_element = driver.find_element(By.ID, 'SIGUNGU_NM0')
        sigungu_select_box = Select(sigungu_select_element)

        # 시/군/구 선택
        sigungu_select_box.select_by_index(j)
        time.sleep(sleep_time)

        # 조회 후 요소들 대기
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIGUNGU_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'totCnt')))
        time.sleep(sleep_time)

        # 주유소 세부정보 링크 객체 수
        tot_cnt = int(driver.find_element(By.ID, 'totCnt').text)
        # 지역의 주유소가 0개면 다음 루프로
        if tot_cnt == 0:
            continue
        
        # 세부정보 요소 대기
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.CSS_SELECTOR, '.rlist > a')))
        time.sleep(sleep_time) # 페이지 요소들 로딩될때까지 명시적으로 잠시 더 대기
    
        # 주유소 세부정보 크롤링
        # 주유소 세부정보 링크 객체 리스트
        href_elements = driver.find_elements(By.CSS_SELECTOR, '.rlist > a')
        

        # 각 주유소 세부정보 링크 클릭 -> 세부정보 크롤링 루프문
        for n in range(tot_cnt):
            try:
                # 링크 클릭
                href_elements[n].click()
                WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'rd_addr')))
                WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'os_nm')))
                WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'b027_p')))
                WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'd047_p')))
                WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'phn_no')))
                time.sleep(sleep_time)

                # Beautiful soup 객체 생성
                results_html = driver.page_source
                soup = BeautifulSoup(results_html, 'html.parser')

                # 주소 문자열 추출 및 리스트 저장
                all_address = soup.select_one('#rd_addr').text
                all_address_lst.append(all_address)

                # 이름 문자열 추출 및 리스트 저장
                all_name = soup.select_one('#os_nm').text
                all_name_lst.append(all_name)

                # 휘발유 가격 추출 및 리스트 저장
                all_g_price = soup.select_one('#b027_p').text
                all_g_price_lst.append(all_g_price)
                
                # 경유 가격 추출 및 리스트 저장
                all_d_price = soup.select_one('#d047_p').text
                all_d_price_lst.append(all_d_price)

                # 전화번호 추출 및 리스트 저장
                all_phone = soup.select_one('#phn_no').text
                all_phone_lst.append(all_phone)
            
            except:
                continue

print(len(all_name_lst))
print(len(all_g_price_lst))
print(len(all_d_price_lst))
print(len(all_address_lst))
print(len(all_phone_lst))
print(all_name_lst[0], all_name_lst[-1])
print(all_g_price_lst[0], all_g_price_lst[-1])
print(all_d_price_lst[0], all_d_price_lst[-1])
print(all_address_lst[0], all_address_lst[-1])
print(all_phone_lst[0], all_phone_lst[-1])



In [29]:
# 전국 주유소 ID 크롤링(주유소 상세정보 API 조회용)

# 명시적 대기 시간
sleep_time = 0.5
# 최대 대기 시간
max_sleep_time = 60

# 전국 주유소 ID 리스트 초기화
all_id_lst = []

# selenium 드라이버 객체 생성 및 페이지 불러오기
driver = webdriver.Chrome()
driver.get('https://www.opinet.co.kr/searRgSelect.do')
WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'VLT_YN')))
WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'searRgSelect')))
time.sleep(sleep_time)

# 불법 주유소 제외 클릭 후 조회 클릭
vlt_yn_element = driver.find_element(By.ID, 'VLT_YN')
vlt_yn_element.click()
sear_element = driver.find_element(By.ID, 'searRgSelect')
sear_element.click()

# 시/도 selectbox의 옵션 수
WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))
time.sleep(sleep_time)
sido_cnt = len(Select(driver.find_element(By.ID, 'SIDO_NM0')).options)


# 전국 주유소 이름, 휘발유 가격, 경유 가격, 주소 크롤링 루프문
for i in range(1, sido_cnt):

    # 시/도 선택하는 select_box 객체 생성
    WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))
    time.sleep(sleep_time)
    sido_select_element = driver.find_element(By.ID, 'SIDO_NM0')
    sido_select_box = Select(sido_select_element)

    # 시/도 선택
    sido_select_box.select_by_index(i)
    WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIGUNGU_NM0')))
    time.sleep(sleep_time)

    # 시/군/구 selectbox의 옵션 수
    sigungu_cnt = len(Select(driver.find_element(By.ID, 'SIGUNGU_NM0')).options)

    for j in range(1, sigungu_cnt):

        # 루프를 continue로 빠져나온 후 대기 시간
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIGUNGU_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'totCnt')))

        # 시/군/구 선택하는 select_box 객체 생성
        time.sleep(sleep_time)
        sigungu_select_element = driver.find_element(By.ID, 'SIGUNGU_NM0')
        sigungu_select_box = Select(sigungu_select_element)

        # 시/군/구 선택
        sigungu_select_box.select_by_index(j)
        time.sleep(sleep_time)

        # 조회 후 요소들 대기
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'SIGUNGU_NM0')))
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.ID, 'totCnt')))
        time.sleep(sleep_time)

        # 주유소 세부정보 링크 객체 수
        tot_cnt = int(driver.find_element(By.ID, 'totCnt').text)
        # 지역의 주유소가 0개면 다음 루프로
        if tot_cnt == 0:
            continue
        
        # 세부정보 요소 대기
        WebDriverWait(driver, max_sleep_time).until(ec.presence_of_element_located((By.CSS_SELECTOR, '.rlist > a')))
        time.sleep(sleep_time) # 페이지 요소들 로딩될때까지 명시적으로 잠시 더 대기
               
        # Beautiful soup 객체 생성
        oil_id_html = driver.page_source
        soup = BeautifulSoup(oil_id_html, 'html.parser')

        # 주유소 ID가 포함된 태그 리스트 생성
        parts_js_lst = soup.select('.rlist')

        # 각 주유소의 ID 추출 루프문
        for idx, part_js in enumerate(parts_js_lst):

            # 모든 주유소의 ID를 추출했다면 반복문 종료
            if idx == tot_cnt - 1:
                break

            # 각 주유소의 태그를 받아서 주유소 ID가 포함된 문자열 리스트 추출
            parts = part_js.select_one('a').attrs['href']
            parts = parts.split("'")

            # 주유소 ID를 받을 문자열 초기화
            oil_id_one = ''

            # 문자열 리스트를 검사하여 주유소 ID를 추출
            for part in parts:
                if len(part) == 8 and part.isalnum() and part[0].isalpha() and part[1].isdigit():
                    oil_id_one = part
                    break
            
            # 추출한 주유소 ID를 리스트에 저장
            all_id_lst.append(oil_id_one)
        



In [ ]:
# 중복 제거
print(len(all_id_lst))

all_id_set = set(all_id_lst)
all_id_lst_dp = list(all_id_set)

print(len(all_id_lst_dp))

11602
10007


In [35]:
# 주유소 id csv파일로 저장
with open('oil_id.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerows([[id] for id in all_id_lst_dp])

In [ ]:
# 주유소 id csv파일에서 다시 파이썬 리스트로
with open('oil_id.csv', 'r', encoding='utf-8', newline='') as f:
    all_id_lst_final = f.readlines()

all_id_lst_final = [id.replace('\r', '').replace('\n', '') for id in all_id_lst_final]
print(all_id_lst_final)

In [ ]:
# api 호출 이후 주유소 컬럼들을 oil_info.csv 파일에 저장
with open('oil_info.csv', 'w', encoding='utf-8-sig', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['o_name', 'o_addr', 'o_phone', 'gasoline_price', 'diesel_price'])
    
    api_keys = [
                
                ] # api 키 7개!

    oil_info_lst = []

    for idx, id in enumerate(all_id_lst_final):
        key_idx = (idx // 1450) % len(api_keys)
        cur_key = api_keys[key_idx]

        url = f'https://www.opinet.co.kr/api/detailById.do?out=json&id={id}&certkey=' + key
        response = requests.get(url)
        response.encoding = 'utf-8'
        response_body = response.json()['RESULT']['OIL'][0]

        g_price, d_price = 0, 0

        for price in response_body['OIL_PRICE']:
            if price['PRODCD'] == 'B027':
                g_price = price['PRICE']
            elif price['PRODCD'] == 'D047':
                d_price = price['PRICE']
            else:
                continue

        oil_info_lst = [response_body['OS_NM'], response_body['NEW_ADR'], response_body['TEL'], g_price, d_price]

        writer.writerow(oil_info_lst)
        time.sleep(0.1)
        
